In [1]:
# STANDIN A0
# Load the data, the split and the shared helpers (parts/00_setup.ipynb). Dropped from the hand-in
# notebook, where the setup cells come first.
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project
Data folder: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project/UNSW-NB15
score() and score_proba() ready


Rows with an infinite or missing value: 0
Loaded 447,915 flows x 76 features


Raw rows:                      447,915
Exact duplicates removed:      141,742
Conflicting rows removed:        1,068
Clean rows (all are used):     305,105


Dropped 9 constant columns: Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, URG Flag Count, CWR Flag Count, ECE Flag Count, Fwd Bytes/Bulk Avg, Fwd Packet/Bulk Avg, Fwd Bulk Rate Avg
67 features: 65 continuous, 2 0/1 columns (Fwd PSH Flags, RST Flag Count)
TUNE_IDX: 36,612 training rows for settings searches
Setup complete: 22 shared names ready in 55 s


In [2]:
# STANDIN B2
# The Part B black box with the settings chosen on validation in B2 (parts/10_baseline_models.ipynb).
# Dropped from the hand-in notebook, where the real B2 cell comes first.
from sklearn.ensemble import HistGradientBoostingClassifier

BOOST_SETTINGS = {"learning_rate": 0.1, "max_leaf_nodes": 31, "random_state": SEED}
def make_boost():
    return HistGradientBoostingClassifier(**BOOST_SETTINGS)

boost = make_boost().fit(X_train, y_train)

<!-- STEP G1 -->
## Part G: adaptability, a new attack the model has never seen

Every test so far used attacks of the same kinds as in training. Real attackers change their methods, and
a model that scored well last month can quietly start missing a new kind of attack. Part G imitates
that on our own data.

**Step G1: hide one attack type.** We remove **every Fuzzers flow from the training set** (and only from
the training set) and train the black box again, with the same settings as in B2 (`make_boost()`). This
model, `boost_hidden`, has never seen a Fuzzers flow. Fuzzing means sending large amounts of random or
malformed data to make a program crash or misbehave.

**Why Fuzzers.** It is the third-largest attack type (15,606 training flows), and in a trial on the
**validation** set, before Part G was written, hiding Fuzzers lowered its recall the most of all attack
types (from 0.989 to 0.696; Exploits, Reconnaissance, DoS, Generic and Shellcode stayed at 0.97 or more).
The choice used no test data.

**What stays where.**
- The validation and test sets are unchanged: they still contain their Fuzzers flows (5,202 each), so we
  can measure how well the model recognises the new attack.
- The removed training flows are kept aside as `X_new, y_new`: the pool of flows an analyst could label in
  step G3, when the model is adapted. **Test-set Fuzzers flows are never used for training.**

In [3]:
# STEP G1
HIDDEN = "Fuzzers"
keep = (type_train != HIDDEN).to_numpy()

X_seen, y_seen = X_train[keep], y_train[keep]              # training set without the hidden type
X_new, y_new = X_train[~keep], y_train[~keep]              # the hidden flows: G3 may label some of them

assert len(X_new) == 15_606 and (y_new == 1).all()
assert (type_train[keep] != HIDDEN).all()
assert int((type_test == HIDDEN).sum()) == 5_202 and int((type_val == HIDDEN).sum()) == 5_202
assert not X_new.index.isin(X_test.index).any() and not X_new.index.isin(X_val.index).any()

boost_hidden = make_boost().fit(X_seen, y_seen)

g1 = pd.DataFrame({"training flows": [len(X_train), len(X_seen)],
                   "of which attacks": [int(y_train.sum()), int(y_seen.sum())],
                   "attack types": [type_train[y_train == 1].nunique(), type_train[keep][y_seen == 1].nunique()],
                   "trees": [boost.n_iter_, boost_hidden.n_iter_]},
                  index=["boost (Part B, all types)", f"boost_hidden (without {HIDDEN})"])
g1.to_csv(TABLES / "G1_hidden_training.csv")
print(f"Removed {len(X_new):,} {HIDDEN} flows from training; they stay in validation and test.")
g1

Removed 15,606 Fuzzers flows from training; they stay in validation and test.


,training flows,of which attacks,attack types,trees
"boost (Part B, all types)",183063,48154,9,100
boost_hidden (without Fuzzers),167457,32548,8,100


<!-- STEP G1 -->
### What G1 set up

| Model | Training flows | Attacks among them | Attack types | Trees |
|---|---:|---:|---:|---:|
| `boost` (Part B, all types) | 183,063 | 48,154 | 9 | 100 |
| `boost_hidden` (without Fuzzers) | 167,457 | 32,548 | 8 | 100 |

The 15,606 Fuzzers flows (a third of all training attacks) are gone from the training data; the model
otherwise has exactly the same settings. Step G2 measures on the test set what this costs: recall on all
attacks, and recall on the 5,202 Fuzzers flows alone, which this model has never seen.